# Convertir cnn_4clases.keras -> TFLite float16
La versión int8 degradaba el modelo (colapsaba a la clase "Otro"). Aquí se convierte el **mismo modelo entrenado** a **float16** (casi sin pérdida). **No re-entrena nada.**

**Antes de correr:** panel derecho → **Add Data → Upload** → sube `cnn_4clases.keras` como dataset, y agrégalo al notebook.


## Convertir


In [ ]:
import os, glob
import tensorflow as tf
from IPython.display import FileLink, display

# 1) Localizar el .keras que subiste como dataset (en /kaggle/input).
rutas = glob.glob("/kaggle/input/**/cnn_4clases.keras", recursive=True)
print("keras encontrado:", rutas)
assert rutas, "No encontré cnn_4clases.keras en /kaggle/input. ¿Lo subiste con Add Data -> Upload y lo agregaste?"
modelo = tf.keras.models.load_model(rutas[0])

# 2) Convertir a float16 (pesos en float16; entrada/salida en float32).
conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tfl = conv.convert()
open("/kaggle/working/triatominos.tflite", "wb").write(tfl)

# 3) Verificar: entrada y salida DEBEN ser float32, salida de longitud 4.
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i = it.get_input_details()[0]; o = it.get_output_details()[0]
print("input  dtype/shape:", i["dtype"], i["shape"])
print("output dtype/shape:", o["dtype"], o["shape"])
print("tamaño MB:", round(len(tfl) / 1e6, 2))
print("CLASES (orden): ['Panstrongylus','Rhodnius','Triatoma','Otro']")
print("Archivos en /kaggle/working:", os.listdir("/kaggle/working"))
display(FileLink("/kaggle/working/triatominos.tflite"))

## Prueba rápida (opcional): ¿discrimina?
Corre esto para confirmar que el modelo NO colapsa a una sola clase.


In [ ]:
import numpy as np
CL = ["Panstrongylus","Rhodnius","Triatoma","Otro"]
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
ii = it.get_input_details()[0]; oo = it.get_output_details()[0]
for nombre, px in [("negro", np.zeros((224,224,3),np.float32)),
                   ("blanco", np.full((224,224,3),255,np.float32)),
                   ("ruido", np.random.randint(0,256,(224,224,3)).astype(np.float32))]:
    it.set_tensor(ii["index"], px[None]); it.invoke()
    p = it.get_tensor(oo["index"])[0]
    print(f"{nombre:7s} -> {CL[int(p.argmax())]:14s}  probs={[round(float(x),2) for x in p]}")